# Inference Pipeline

# DISTILLBERT

In [1]:
import os
import json
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
import sys

# Add parent directory to sys.path so we can import from the project root (useful for Jupyter or script)
# os.getcwd()        → returns current working directory, e.g., "/path/to/symptom-ner/v01"
# os.path.join(..., "..") → moves one directory up, i.e., "/path/to/symptom-ner"
# os.path.abspath()  → resolves this to the absolute path
PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

from gcp_utils import download_from_gcs, list_bucket_files
from config import settings


# ------- Load labels and test data - LOCALLY -------
with open("data/distillbert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("data/label2id.json", "r") as f:
    label2id = json.load(f)
# ------------------------------------------------------

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


# CONFIG FOR LOADING FROM GCS 

# v01/runs/distilbert-base-uncased/run_0/
VERSION = "v01"
MODEL_NAME = "distilbert-base-uncased"  # or "dmis-lab/biobert-base-cased-v1.2" for BioBERT
RUN_IDX = 0  

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"


/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()

✅ Model found locally at ./downloaded_models/distilbert-base-uncased/run_0
Skipping download from GCS.
📂 Loading model from ./downloaded_models/distilbert-base-uncased/run_0...
Using device: mps


DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

# **Token Level Prediction**

In [3]:
from inference_utils import predict_token_level
test_text = "Patient reports severe headache and nausea"
tokens, predictions = predict_token_level(test_text, model, tokenizer, device=device)
print(f"Tokens {len(tokens)}:\n\t{tokens}")
print(f"Predictions {len(predictions)}:\n\t{predictions}")

Tokens 6:
	['patient', 'reports', 'severe', 'headache', 'and', 'nausea']
Predictions 6:
	[4, 4, 1, 3, 3, 3]


In [4]:
# RUN ANOTHER EXAMPLE: 
sample = json.loads(test_data[1])
text = sample.get('text')
tokens = sample.get('tokens')
token_label_ids = sample.get('token_label_ids')
print(f"TEXT: {text}")
print(f"TOKENS from test data: {tokens}")
tks, predictions = predict_token_level(text, model, tokenizer, device=device)
print(f"Returned tokens: {tks}")
print(f"Predictions: {predictions[0]}")


TEXT: The patient has lymphatic system symptom.
TOKENS from test data: ['[CLS]', 'the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.', '[SEP]']
Returned tokens: ['the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.']
Predictions: 4


# **Word Level Prediction**

In [5]:
from inference_utils import predict_word_level

sample = json.loads(test_data[1])
text = sample.get('text')

predict_word_level(text=text, model=model, tokenizer=tokenizer, id2label=id2label, device=device)

(['the',
  'patient',
  'has',
  'l',
  '##ym',
  '##pha',
  '##tic',
  'system',
  'sy',
  '##mpt',
  '##om',
  '.'],
 ['O',
  'O',
  'O',
  'B-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'O'],
 [0, 1, 2, 3, 3, 3, 3, 4, 5, 5, 5, 6],
 ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom', '.'],
 ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O'])

In [6]:
from inference_utils import predict_word_level

#sample = json.loads(test_data[1])
text = "The patient has cataplexy." #lymphatic system symptom and cataplexy."#sample.get('text')
samples = ["The patient has cataplexy.", "The patient has lymphatic system symptom.", "The patient has lymphatic system symptom and cataplexy.","The patient has lymphatic system symptom.", "The patient has cataplexy and lymphatic system symptom.", "Roberta does not have back pain but she has an inflamation in her wrist"]
for s in samples:
    print("="*20)
    print(f" TEXT: {s}")
    print("="*20)
    tokens,token_labels,word_ids, words, word_labels =  predict_word_level(text=s, model=model, tokenizer=tokenizer, id2label=id2label, device=device)
    print("tokens: ", tokens)
    print("token labels: ", token_labels)
    print("word_ids: ", word_ids)
    print("word: ", words)
    print("bio word labels: ", word_labels)
    print()

 TEXT: The patient has cataplexy.
tokens:  ['the', 'patient', 'has', 'cat', '##ap', '##le', '##xy', '.']
token labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
word_ids:  [0, 1, 2, 3, 3, 3, 3, 4]
word:  ['The', 'patient', 'has', 'cataplexy', '.']
bio word labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'O']

 TEXT: The patient has lymphatic system symptom.
tokens:  ['the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.']
token labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
word_ids:  [0, 1, 2, 3, 3, 3, 3, 4, 5, 5, 5, 6]
word:  ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom', '.']
bio word labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']

 TEXT: The patient has lymphatic system symptom and cataplexy.
tokens:  ['the', 'patient', 'has', 'l', '##ym', 

# Test word_labels --> entity_spans

In [52]:
examples = {
    "ex1" : [['The', 'patient', 'has', 'lymphatic', 'system', 'symptom', '.'],
    ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
    ],
    "ex2" : [
        ['The', 'patient', 'has', 'cataplexy', '.'],
        ['O', 'O', 'O', 'B-SYMPTOM_POS', 'O']
        
    ],
    "ex3" : [
        ['Roberta', 'does', 'not', 'have', 'back', 'pain', 'but', 'she', 'has', 'an', 'inflamation', 'in', 'her', 'wrist'],
        ['O', 'O', 'O', 'O', 'B-SYMPTOM_NEG', 'I-SYMPTOM_NEG', 'O', 'O', 'O', 'O', 'I-SYMPTOM_POS', 'I-SYMPTOM_NEG', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS']
    ]

    
}


In [ ]:
from typing import List, Dict

# TODO:
# read and understad code + mercor 17

def word_labels_to_spans(words: List[str], word_labels: List[str]) -> List[Dict]:
    """
    Convert word-level BIO labels into character-level spans over the joined text.

    - Input labels are like: "O", "B-SYMPTOM_POS", "I-SYMPTOM_POS", "B-SYMPTOM_NEG", ...
    - Output spans use **character offsets** into: text = " ".join(words)

    Behavior:
    - "O" becomes its own single-word span (per your earlier behavior).
      (If you want to drop O spans, we can remove that.)
    - "B-XXX" starts a new entity span of type XXX.
    - "I-XXX" continues the current entity if it matches XXX; otherwise it starts a new entity
      (robust to "I" appearing without a matching previous "B").
    """

    # Sanity check: labels must align 1:1 with words
    assert len(words) == len(word_labels), "words and word_labels must be same length"

    # This is the text we will slice spans from using char indices
    text = " ".join(words).strip()

    spans: List[Dict] = []

    # -------------------------------------------------------------------------
    # Step 1) Precompute the character [start, end) offsets for every word
    #
    # Example: words = ["The", "patient", "."]
    # text = "The patient ."
    # positions = [(0,3), (4,11), (12,13)]
    #
    # We do this once so we DON'T have to manually update indices inside the BIO logic.
    # -------------------------------------------------------------------------
    positions = []
    pos = 0
    for w in words:
        start = pos
        end = start + len(w)
        positions.append((start, end))
        pos = end + 1  # +1 to skip the space between words in " ".join(words)

    # -------------------------------------------------------------------------
    # Step 2) Track the "current" entity we are building as we scan tokens.
    # If current_label is None, we are not currently inside an entity.
    # -------------------------------------------------------------------------
    current_label = None    # normalized label like "SYMPTOM_POS" (no "B-" / "I-")
    current_start = None    # char start of the entity in `text`
    current_end = None      # char end (exclusive) of the entity in `text`

    # Helper: when an entity ends, emit it into spans and clear state
    def flush_current():
        # `nonlocal` allows this nested function to modify variables from the outer function's scope.
        # Without it, assigning to these variables would create new local variables instead of
        # modifying the outer function's current_label, current_start, and current_end.
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            spans.append({
                "start": current_start,
                "end": current_end,
                "text": text[current_start:current_end],
                "label": current_label,
            })
            current_label = None
            current_start = None
            current_end = None

    # -------------------------------------------------------------------------
    # Step 3) Scan tokens and apply BIO rules (state machine)
    # -------------------------------------------------------------------------
    for (word, raw_label), (w_start, w_end) in zip(zip(words, word_labels), positions):

        # Case A) Outside: close any open entity, and optionally emit an "O" span
        if raw_label == "O":
            flush_current()
            spans.append({
                "start": w_start,
                "end": w_end,
                "text": text[w_start:w_end],
                "label": "O",
            })
            continue

        # For BIO labels like "B-SYMPTOM_POS" / "I-SYMPTOM_POS":
        # prefix = "B" or "I"
        # norm   = "SYMPTOM_POS" (everything after the first "-")
        prefix, norm = raw_label.split("-", 1)

        # Case B) Begin: always start a new entity (but flush any previous first)
        if prefix == "B":
            flush_current()
            current_label = norm
            current_start = w_start
            current_end = w_end

        # Case C) Inside: extend if it matches; otherwise start a new one (robust behavior)
        elif prefix == "I":
            if current_label == norm:
                # Same entity continues: just extend the end pointer
                current_end = w_end
            else:
                # Mismatch or "I" without prior "B": treat as a new entity start
                flush_current()
                current_label = norm
                current_start = w_start
                current_end = w_end

        # Case D) Unexpected label format: treat as a standalone span
        else:
            flush_current()
            spans.append({
                "start": w_start,
                "end": w_end,
                "text": text[w_start:w_end],
                "label": raw_label,
            })

    # If we ended while still inside an entity, append the last entity span
    flush_current()
    return spans

word_labels_to_spans(examples["ex1"][0], examples["ex1"][1])

[{'start': 0, 'end': 3, 'text': 'The', 'label': 'O'},
 {'start': 4, 'end': 11, 'text': 'patient', 'label': 'O'},
 {'start': 12, 'end': 15, 'text': 'has', 'label': 'O'},
 {'start': 16,
  'end': 40,
  'text': 'lymphatic system symptom',
  'label': 'SYMPTOM_POS'},
 {'start': 41, 'end': 42, 'text': '.', 'label': 'O'}]

In [ ]:
word_labels_to_spans(examples["ex2"][0], examples["ex2"][1])

's'

In [16]:
t = "The patient has cataplexy ."
t[12:15]

'has'

In [ ]:
words = ['brazil', 'is']#, 'in','america','south', 'america']
text = " ".join(words).strip()

start_index = 0
for i, word in enumerate(words):
    print(f"starting index of word {word}: {start_index}")
    end_index = start_index + len(word) # dont put - 1 because then it is easier for slicing
    print(f"ending index of word {word} (+1): {end_index}")
    start_index += len(word) + 1 # to account for space


starting index of word brazil: 0
ending index of word brazil: 6
starting index of word is: 7
ending index of word is: 9


In [ ]:
"B-SYMPTOM_POS".split("-")[1]

'SYMPTOM_POS'

In [ ]:
t = "brazil"
len(t)

6

In [ ]:

words = ['brazil', 'is', 'in', 'south', 'america']
text = " ".join(words).strip()

def word_start_indices(words, text):
    """
    Returns a list of the starting character indices for each word in the words list,
    corresponding to their first character's position in the full text string.

    Logic:
    - We iterate through the list of words in order.
    - For each word, we search for its first occurrence in the `text` string, 
      starting from the position after the previous word.
    - We use text.find(word, current_pos), which looks for `word` in `text`
      at or after the index `current_pos` and returns the starting character index
      of that occurrence. If the word is not found, it returns -1.
    - For each found word, we record its starting character index,
      and then advance our cursor so that matches do not overlap.

    Example:
        'brazil is in south america'
         0123456 7 89  13   19
         ^      ^ ^   ^    ^
         0      7 10  13   19

    Returns:
        List of ints. Ex: [0, 7, 10, 13, 19]
    """
    indices = []
    current_pos = 0
    for word in words:
        idx = text.find(word, current_pos)
        if idx == -1:
            raise ValueError(f"Word '{word}' not found in text starting from position {current_pos} in '{text}'")
        indices.append(idx)
        # Move cursor past this word so future finds are correct
        current_pos = idx + len(word)
    return indices

start_indices = word_start_indices(words, text)

In [ ]:
text

'brazil is in south america'